# 02 – Mini-RAG: retrieval dhe krahasimi LLM pa/me fragmente (Detyrat C dhe D)
Projekti 06 • Java 2 • Roli B (retrieval/LLM) + Roli C (kontrolli manual)

Rrjedha: **pyetje → top-3 fragmente (FAISS) → (a) LLM pa fragmente = baseline, (b) i njëjti LLM me fragmentet = RAG
→ kontroll automatik i citimit → kontroll manual.**

Modeli dhe parametrat lexohen nga `config/llm_config.json`. Pa çelës API përdoret `stub_extractive`, i cili
**nuk është LLM** dhe shërben vetëm për të provuar tubacionin.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path("..").resolve() / "src"))
import minirag as mr
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)

# Korpusi: REAL06 nëse ekzistojnë abstraktet reale (pas `python src/merr_abstraktet.py`), përndryshe SIM06.
REAL_PATH = mr.ROOT / "data/private/02_Abstraktet_Reale.csv"
CORPUS = "REAL06" if REAL_PATH.exists() else "SIM06"
print("Korpusi aktiv:", CORPUS)

import run_java02 as runner
cfg = mr.load_llm_config()
llm = mr.LLMClient(cfg)
print("LLM:", llm.model_version)
print("Parametrat (të njëjtë për baseline dhe RAG):", json.dumps(llm.settings(), ensure_ascii=False))

Korpusi aktiv: REAL06
LLM: stub_extractive (JO LLM)
Parametrat (të njëjtë për baseline dhe RAG): {"provider": "stub (mungon GROQ_API_KEY)", "model": "stub_extractive (JO LLM)", "temperature": 0.0, "max_tokens": 300, "seed": 42, "top_p": 1.0}


## 1. Pyetjet e provës

In [2]:
docs, qs, chunks = runner.load(CORPUS)
rag = mr.MiniRAG.build(chunks)
print("Embedder:", rag.embedder.backend, "| Indeksi:", rag.index.backend)
print(qs[["question_id", "question", "expected_behavior"]].to_string(index=False))

Traceback (most recent call last):
  File "/home/claude/projekti06/tools/build_notebooks.py", line 43, in execute
    exec(compile(c["source"], f"{name}[{n}]", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "02_Mini_RAG_Prova.ipynb[2]", line 1, in <module>
  File "/home/claude/projekti06/src/run_java02.py", line 41, in load
    docs = mr.load_real_corpus()
  File "/home/claude/projekti06/src/minirag.py", line 64, in load_real_corpus
    df = pd.read_csv(path)
  File "/usr/local/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/usr/local/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/usr/local/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 1645, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^

## 2. Retrieval: top-3 fragmente për çdo pyetje, krahasuar me kërkimin me fjalë kyçe

In [3]:
for q in qs.itertuples():
    top = rag.retrieve(q.question, k=3)
    kw = mr.keyword_search(chunks, q.question, k=3)
    abst, rule = mr.needs_abstention(q.question, top)
    print(f"\n{q.question_id}: {q.question}")
    for r in top.itertuples():
        mark = "  ← burimi i pritur" if r.source_id == q.expected_source_id else ""
        txt = r.chunk_text if r.shareable else "[tekst vetëm lokal]"
        print(f"   #{r.rank} {r.chunk_id} score={r.retrieval_score:.3f}{mark}\n       {txt[:160]}")
    print("   fjalë kyçe top-3:", [f"{c} ({n} fjalë)" for c, n in kw])
    print("   abstention:", abst, rule)

Traceback (most recent call last):
  File "/home/claude/projekti06/tools/build_notebooks.py", line 43, in execute
    exec(compile(c["source"], f"{name}[{n}]", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "02_Mini_RAG_Prova.ipynb[3]", line 1, in <module>
NameError: name 'qs' is not defined


**Leximi i rezultatit:** pikëzimi (score) nuk është provë. Rishikuesi lexon tekstin e fragmentit dhe plotëson
kolonën `manual_relevance` te `results/02_Retrieval_Prova.csv` (përkatës / pjesërisht / jo përkatës).

## 3. Prompt-et: e njëjta pyetje, e njëjta LLM, me dhe pa fragmente

In [4]:
q0 = qs.iloc[1]
top0 = rag.retrieve(q0.question, k=3)
prompts = mr.build_prompts(q0.question, top0)
print("SYSTEM:", mr.SYSTEM_PROMPT, "\n")
print("----- BASELINE -----\n" + prompts["baseline"])
print("\n----- RAG -----\n" + runner.redact_prompt(prompts["rag"], chunks))

Traceback (most recent call last):
  File "/home/claude/projekti06/tools/build_notebooks.py", line 43, in execute
    exec(compile(c["source"], f"{name}[{n}]", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "02_Mini_RAG_Prova.ipynb[4]", line 1, in <module>
NameError: name 'qs' is not defined


## 4. Ekzekutimi i plotë dhe ruajtja e `02_Retrieval_Prova.csv` + `02_Krahasimi_LLM.csv`

In [5]:
import datetime as dt
run_id = dt.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = mr.ROOT / "results/02_run_log.json"
log = json.loads(log_path.read_text(encoding="utf-8")) if log_path.exists() else {"runs": {}}
runner.run(CORPUS, llm, run_id, log)
log_path.write_text(json.dumps(log, ensure_ascii=False, indent=2), encoding="utf-8")

Traceback (most recent call last):
  File "/home/claude/projekti06/tools/build_notebooks.py", line 43, in execute
    exec(compile(c["source"], f"{name}[{n}]", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "02_Mini_RAG_Prova.ipynb[5]", line 5, in <module>
  File "/home/claude/projekti06/src/run_java02.py", line 55, in run
    docs, qs, chunks = load(corpus)
                       ~~~~^^^^^^^^
  File "/home/claude/projekti06/src/run_java02.py", line 41, in load
    docs = mr.load_real_corpus()
  File "/home/claude/projekti06/src/minirag.py", line 64, in load_real_corpus
    df = pd.read_csv(path)
  File "/usr/local/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/usr/local/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/usr/local/lib/python3.13/site-packages/pandas/io/parsers/

## 5. Baseline kundrejt RAG dhe kontrolli automatik i citimit

In [6]:
cmp = pd.read_csv(mr.ROOT / "results/02_Krahasimi_LLM.csv")
cmp = cmp[cmp.corpus == CORPUS]
for qid, g in cmp.groupby("question_id", sort=False):
    print(f"\n{qid}: {g.question_text.iloc[0]}")
    for r in g.itertuples():
        print(f"   [{r.variant:8}] {str(r.output)[:230]}")
        print(f"              citim: {r.citation_check}")

## 6. Kontrolli manual (Roli C – Evis Bakiu)
Kontrolli automatik provon vetëm që (1) ID e cituar ishte mes fragmenteve të dërguara dhe (2) teksti në thonjëza
gjendet fjalë për fjalë. **Nuk provon që pohimi është i saktë në kuptim.**

Për të paktën një citim:
1. Hapni `results/02_Krahasimi_LLM.csv`, rreshtin RAG të pyetjes.
2. Hapni fragmentin te `results/02_Chunket_Pilot.csv` sipas `cited_ids` (ose burimin origjinal nga `source_url`).
3. Plotësoni `manual_verdict` = *e mbështetur / pjesërisht / e pambështetur / abstention e saktë* dhe `manual_note`.
4. Për baseline: shënoni çdo pohim ose referencë pa bazë si *e pambështetur*.

In [7]:
pending = cmp[cmp.manual_verdict.isna()][["question_id", "variant", "citation_check"]]
print(f"Rreshta që presin kontroll manual: {len(pending)}")
print(pending.to_string(index=False))

Rreshta që presin kontroll manual: 0
Empty DataFrame
Columns: [question_id, variant, citation_check]
Index: []
